# VRA/ANAC — ELT de 2020
**Projeto de Integração — Banco de Dados · CIn/UFPE**

**Adaptação para 2020:** Gustavo José Arraes de Queiroz.  
**Base:** notebook `notebooks/elt/elt_2019.ipynb` desenvolvido pelo grupo, no [repositório do projeto](https://github.com/LeonardoBarrosCavalcantiNeto/Projeto-de-integracao-BD).

O ELT primeiro carrega os valores originais no PostgreSQL; só depois o SQL trata duplicatas, campos vazios, datas e métricas.

**Como usar:** no Google Colab, execute as células na ordem. Na etapa 3, envie `data_2020_anac.zip`, com os 12 CSVs de janeiro a dezembro. Pode haver uma pasta dentro do ZIP. Não é necessário abrir ou salvar os CSVs no Excel.

**Escopo:** carga bruta, transformação, auditoria e exportação de evidências de 2020. A carga dimensional e a comparação ETL × ELT estão no [guia da integração final](../../documentacao/INTEGRACAO_FINAL.md).

**Registro da etapa individual:** adaptação verificada estaticamente. A execução completa desta versão no Colab segue pendente. As saídas antigas foram removidas. As células de validação abaixo só indicarão sucesso depois da execução.

> O PostgreSQL será temporário, dentro do Colab. Ao encerrar a sessão, o banco pode desaparecer. Baixe as evidências que quiser conservar.

## 1. Instalar o PostgreSQL
Execute e espere terminar. Isso prepara o servidor e a biblioteca de conexão no Colab.

In [ ]:
import os
import subprocess
import sys

from IPython import get_ipython

shell = get_ipython()
if (shell is None or not type(shell).__module__.startswith("google.colab.")
        or not os.environ.get("COLAB_RELEASE_TAG")):
    raise RuntimeError("Execute este notebook no Colab hospedado. Para a execução local, siga o README.")

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "postgresql", "postgresql-contrib", "-qq"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psycopg2-binary", "pandas>=2.2,<3.0"], check=True)
subprocess.run(["service", "postgresql", "start"], check=True)
print("PostgreSQL instalado e iniciado no Colab.")

## 2. Conectar ao banco
A senha é temporária e não é exibida. Usaremos apenas o schema `elt_2020`.

In [ ]:
import secrets
import psycopg2
import subprocess

senha_pg = secrets.token_urlsafe(24)
# Envía a senha ao psql por stdin (evita registrá-la na linha de comando)
subprocess.run(
    ["runuser", "-u", "postgres", "--", "psql", "-v", "ON_ERROR_STOP=1"],
    input="ALTER USER postgres PASSWORD '" + senha_pg.replace("'", "''") + "';\n",
    text=True, check=True, capture_output=True
)
conn = psycopg2.connect(host="localhost", port=5432, dbname="postgres", user="postgres", password=senha_pg)
conn.autocommit = False
print("Conexão PostgreSQL funcionando!")

## 3. Enviar o ZIP de 2020
Execute e clique em **Escolher arquivos/Procurar**. Selecione somente `data_2020_anac.zip`. Aguarde o envio terminar.

In [ ]:
validacao_concluida = False
from google.colab import files
import io
import csv
import zipfile
import hashlib
import re
from pathlib import PurePosixPath
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display
from psycopg2 import sql

ANO = 2020
SCHEMA = "elt_2020"
VERSAO_PIPELINE = "1.0"
inicio_execucao = datetime.now(timezone.utc).isoformat()

arquivos_enviados = files.upload()
assert len(arquivos_enviados) == 1, "Envie somente um ZIP com os 12 CSVs de 2020."
nome_zip, bytes_zip = next(iter(arquivos_enviados.items()))
assert nome_zip.lower().endswith(".zip"), "O arquivo enviado precisa ser ZIP."
assert zipfile.is_zipfile(io.BytesIO(bytes_zip)), "O conteúdo enviado não é um ZIP válido."
sha256_zip = hashlib.sha256(bytes_zip).hexdigest()
print("ZIP recebido:", nome_zip)
print("SHA-256:", sha256_zip)

## 4. Conferir os 12 arquivos sem modificar os dados
Conferimos nomes, cabeçalhos e número de campos. Cada mês precisa aparecer uma vez. Todos os valores continuam como texto.

`arquivo_origem` identifica o CSV e `linha_origem` indica a linha física onde o registro começa, contando o cabeçalho como linha 1. A leitura respeita campos entre aspas, inclusive quebras de linha dentro deles.

In [ ]:
colunas_esperadas = [
    "Sigla ICAO Empresa Aérea", "Empresa Aérea", "Número Voo", "Código DI",
    "Código Tipo Linha", "Modelo Equipamento", "Número de Assentos",
    "Sigla ICAO Aeroporto Origem", "Descrição Aeroporto Origem", "Partida Prevista",
    "Partida Real", "Sigla ICAO Aeroporto Destino", "Descrição Aeroporto Destino",
    "Chegada Prevista", "Chegada Real", "Situação Voo", "Justificativa",
    "Referência", "Situação Partida", "Situação Chegada"
]

def registros_csv(arquivo_binario, nome):
    leitor = csv.reader(io.TextIOWrapper(arquivo_binario, encoding="utf-8-sig", newline=""),
                        delimiter=";", strict=True)
    cabecalho = next(leitor, None)
    if cabecalho != colunas_esperadas:
        raise ValueError(f"Cabeçalho diferente do contrato em {nome}: {cabecalho!r}")
    while True:
        inicio_linha = leitor.line_num + 1
        try:
            campos = next(leitor)
        except StopIteration:
            break
        if len(campos) != 20:
            raise ValueError(f"{nome}, linha {inicio_linha}: esperados 20 campos, encontrados {len(campos)}.")
        yield inicio_linha, campos

inventario = []
with zipfile.ZipFile(io.BytesIO(bytes_zip)) as z:
    membros = [m for m in z.infolist() if not m.is_dir()
               and not m.filename.startswith("__MACOSX/")
               and m.filename.lower().endswith(".csv")]
    assert len(membros) == 12, f"Esperados 12 CSVs; encontrados {len(membros)}."
    meses = set()
    for membro in sorted(membros, key=lambda m: m.filename):
        nome = PurePosixPath(membro.filename.replace("\\", "/")).name
        correspondencia = re.fullmatch(r"VRA_2020_(0[1-9]|1[0-2])\.csv", nome, flags=re.I)
        assert correspondencia, f"Nome inesperado: {nome}. Use os CSVs originais de 2020."
        mes = int(correspondencia.group(1))
        assert mes not in meses, f"Mês repetido no ZIP: {mes:02d}."
        meses.add(mes)
        assert membro.file_size > 0, f"CSV vazio: {nome}."
        with z.open(membro) as arquivo:
            quantidade = sum(1 for _ in registros_csv(arquivo, nome))
        assert quantidade > 0, f"Nenhum registro em {nome}."
        with z.open(membro) as arquivo:
            sha256 = hashlib.file_digest(arquivo, "sha256").hexdigest()
        inventario.append({"membro_zip": membro.filename, "arquivo_origem": nome,
                           "mes": mes, "registros": quantidade, "bytes": membro.file_size,
                           "sha256": sha256})
assert meses == set(range(1, 13)), "Faltam meses de 2020."
inventario.sort(key=lambda item: item["mes"])
total_csv = sum(item["registros"] for item in inventario)
display(pd.DataFrame(inventario).drop(columns=["membro_zip", "sha256"]))
print(f"Conferência concluída: 12 meses, {total_csv:,} registros. Nenhum valor foi transformado.")

## 5. Criar a tabela bruta
**Ao repetir esta célula, as tabelas deste pipeline no schema `elt_2020` serão reconstruídas.** O schema de 2019 e os demais schemas não são alterados.

As 20 colunas da ANAC são `TEXT`. A chave `(arquivo_origem, linha_origem)` identifica cada ocorrência, inclusive duplicatas.

In [ ]:
validacao_concluida = False
tabelas_pipeline = [
    "resumo_qualidade", "vra_anomalias", "vra_tratada", "vra_rejeicoes",
    "vra_duplicatas_excluidas", "vra_avaliada", "vra_convertida",
    "vra_classificada", "inventario_arquivos", "vra_bruta"
]
with conn:
    with conn.cursor() as cur:
        cur.execute("CREATE SCHEMA IF NOT EXISTS elt_2020")
        for tabela in tabelas_pipeline:
            cur.execute(sql.SQL("DROP TABLE IF EXISTS elt_2020.{}").format(sql.Identifier(tabela)))
        campos = [sql.SQL("{} TEXT NOT NULL").format(sql.Identifier(c)) for c in colunas_esperadas]
        cur.execute(sql.SQL("""
            CREATE TABLE elt_2020.vra_bruta (
                {},
                arquivo_origem TEXT NOT NULL,
                linha_origem INTEGER NOT NULL,
                PRIMARY KEY (arquivo_origem, linha_origem)
            )
        """).format(sql.SQL(", ").join(campos)))
        cur.execute("""
            CREATE TABLE elt_2020.inventario_arquivos (
                arquivo_origem TEXT PRIMARY KEY, mes INTEGER NOT NULL,
                registros BIGINT NOT NULL, bytes BIGINT NOT NULL, sha256 TEXT NOT NULL
            )
        """)
print("Tabela elt_2020.vra_bruta pronta.")

## 6. Carregar os valores originais — Load
O Python transporta os registros. Não aplica `strip`, conversão de datas, remoção de duplicatas ou filtros.

O `COPY` usa aspas em todos os campos, preservando textos vazios e valores que se pareçam com marcadores de nulo. A carga inteira é uma transação: em caso de erro, ela é desfeita.

In [ ]:
colunas_carga = colunas_esperadas + ["arquivo_origem", "linha_origem"]
comando_copy = sql.SQL(
    "COPY elt_2020.vra_bruta ({}) FROM STDIN WITH (FORMAT CSV, NULL '\\N')"
).format(sql.SQL(", ").join(sql.Identifier(c) for c in colunas_carga)).as_string(conn)

with conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM elt_2020.vra_bruta")
        assert cur.fetchone()[0] == 0, "A tabela já contém dados. Para reconstruir a carga, execute a etapa 5."
        with zipfile.ZipFile(io.BytesIO(bytes_zip)) as z:
            for item in inventario:
                buffer = io.StringIO(newline="")
                escritor = csv.writer(buffer, quoting=csv.QUOTE_ALL, lineterminator="\n")
                with z.open(item["membro_zip"]) as arquivo:
                    for linha, campos in registros_csv(arquivo, item["arquivo_origem"]):
                        escritor.writerow(campos + [item["arquivo_origem"], linha])
                buffer.seek(0)
                cur.copy_expert(comando_copy, buffer)
                buffer.close()
                cur.execute("INSERT INTO elt_2020.inventario_arquivos VALUES (%s,%s,%s,%s,%s)",
                            (item["arquivo_origem"], item["mes"], item["registros"], item["bytes"], item["sha256"]))
                print("Carregado:", item["arquivo_origem"])
print("Carga bruta concluída.")

## 7. Conferir a carga
Comparamos a quantidade carregada por arquivo com a quantidade efetivamente lida no ZIP. Não usamos os totais de 2019 como condição de sucesso.

In [ ]:
def consultar(comando, parametros=None):
    with conn:
        with conn.cursor() as cur:
            cur.execute(comando, parametros)
            return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

carga = consultar("""
    SELECT i.arquivo_origem, i.registros AS linhas_csv, COUNT(b.arquivo_origem) AS linhas_banco
    FROM elt_2020.inventario_arquivos i
    LEFT JOIN elt_2020.vra_bruta b USING (arquivo_origem)
    GROUP BY i.arquivo_origem, i.registros
    ORDER BY i.arquivo_origem
""")
display(carga)
assert len(carga) == 12
assert (carga["linhas_csv"] == carga["linhas_banco"]).all(), "A contagem da carga diverge dos CSVs."
assert int(carga["linhas_banco"].sum()) == total_csv
print("Carga bruta conferida.")

## 8. Transformar dentro do PostgreSQL — Transform
O SQL executa, nesta ordem:

1. Compara os **20 valores originais**, preserva a primeira ocorrência por arquivo/linha e separa as duplicatas excedentes.
2. Padroniza campos conforme o contrato: códigos em maiúsculas; nomes e descrições mantêm a grafia; vazios viram `NULL`.
3. Valida datas de verdade, inclusive dias e meses. Se um dos cinco campos obrigatórios estiver ausente ou inválido, registra **todos** os motivos de rejeição.
4. Calcula diferenças com sinal e atrasos não negativos. **Sem horário real, diferença e atraso ficam `NULL`.**
5. Marca diferenças absolutas acima de 24 horas, sem excluir ou corrigir o valor original.

Datas são `TIMESTAMP WITHOUT TIME ZONE`: não presumimos uma conversão de fuso. Anos distantes em datas reais ficam preservados quando válidos no calendário e são evidenciados pelas métricas. `ano_origem` é o ano dos arquivos, não necessariamente o ano de cada partida/chegada.

A tabela bruta permanece intacta. Campos opcionais inválidos são auditados; não tornam o voo automaticamente rejeitado.

In [ ]:
validacao_concluida = False
SQL_TRANSFORMACAO = r"""
-- ELT 2020. Executar depois da carga bruta; nunca altera os 20 campos brutos.
CREATE OR REPLACE FUNCTION elt_2020.limpar_texto(valor TEXT)
RETURNS TEXT LANGUAGE SQL IMMUTABLE STRICT AS $$
    SELECT NULLIF(regexp_replace(valor, '^\s+|\s+$', '', 'g'), '');
$$;

CREATE OR REPLACE FUNCTION elt_2020.datahora_segura(valor TEXT)
RETURNS TIMESTAMP LANGUAGE plpgsql IMMUTABLE STRICT AS $$
BEGIN
    IF valor !~ '^[0-9]{2}/[0-9]{2}/[0-9]{4} [0-9]{2}:[0-9]{2}$' THEN
        RETURN NULL;
    END IF;
    -- make_timestamp valida o calendário; não ajusta 31/02 silenciosamente.
    IF substring(valor,12,2)::INTEGER > 23 OR substring(valor,15,2)::INTEGER > 59 THEN
        RETURN NULL;
    END IF;
    RETURN make_timestamp(substring(valor,7,4)::INTEGER, substring(valor,4,2)::INTEGER,
                          substring(valor,1,2)::INTEGER, substring(valor,12,2)::INTEGER,
                          substring(valor,15,2)::INTEGER, 0);
EXCEPTION WHEN datetime_field_overflow OR invalid_datetime_format OR numeric_value_out_of_range THEN
    RETURN NULL;
END;
$$;

CREATE OR REPLACE FUNCTION elt_2020.data_segura(valor TEXT)
RETURNS DATE LANGUAGE plpgsql IMMUTABLE STRICT AS $$
BEGIN
    IF valor !~ '^[0-9]{4}-[0-9]{2}-[0-9]{2}$' THEN RETURN NULL; END IF;
    RETURN make_date(substring(valor,1,4)::INTEGER, substring(valor,6,2)::INTEGER,
                     substring(valor,9,2)::INTEGER);
EXCEPTION WHEN datetime_field_overflow OR invalid_datetime_format OR numeric_value_out_of_range THEN
    RETURN NULL;
END;
$$;

CREATE OR REPLACE FUNCTION elt_2020.inteiro_seguro(valor TEXT)
RETURNS INTEGER LANGUAGE plpgsql IMMUTABLE STRICT AS $$
BEGIN
    IF valor !~ '^[0-9]+$' THEN RETURN NULL; END IF;
    RETURN valor::INTEGER;
EXCEPTION WHEN numeric_value_out_of_range THEN RETURN NULL;
END;
$$;

DROP TABLE IF EXISTS elt_2020.resumo_qualidade;
DROP TABLE IF EXISTS elt_2020.vra_anomalias;
DROP TABLE IF EXISTS elt_2020.vra_tratada;
DROP TABLE IF EXISTS elt_2020.vra_rejeicoes;
DROP TABLE IF EXISTS elt_2020.vra_duplicatas_excluidas;
DROP TABLE IF EXISTS elt_2020.vra_avaliada;
DROP TABLE IF EXISTS elt_2020.vra_convertida;
DROP TABLE IF EXISTS elt_2020.vra_classificada;

CREATE TABLE elt_2020.vra_classificada AS
SELECT b.*,
       ROW_NUMBER() OVER w AS ocorrencia_num,
       FIRST_VALUE(arquivo_origem) OVER w AS arquivo_preservado,
       FIRST_VALUE(linha_origem) OVER w AS linha_preservada
FROM elt_2020.vra_bruta b
WINDOW w AS (
    PARTITION BY "Sigla ICAO Empresa Aérea" COLLATE "C", "Empresa Aérea" COLLATE "C", "Número Voo" COLLATE "C", "Código DI" COLLATE "C", "Código Tipo Linha" COLLATE "C", "Modelo Equipamento" COLLATE "C", "Número de Assentos" COLLATE "C", "Sigla ICAO Aeroporto Origem" COLLATE "C", "Descrição Aeroporto Origem" COLLATE "C", "Partida Prevista" COLLATE "C", "Partida Real" COLLATE "C", "Sigla ICAO Aeroporto Destino" COLLATE "C", "Descrição Aeroporto Destino" COLLATE "C", "Chegada Prevista" COLLATE "C", "Chegada Real" COLLATE "C", "Situação Voo" COLLATE "C", "Justificativa" COLLATE "C", "Referência" COLLATE "C", "Situação Partida" COLLATE "C", "Situação Chegada" COLLATE "C"
    ORDER BY arquivo_origem COLLATE "C", linha_origem
);

CREATE TABLE elt_2020.vra_duplicatas_excluidas AS
SELECT * FROM elt_2020.vra_classificada WHERE ocorrencia_num > 1;

-- Só após identificar duplicatas EXATAS é feita a padronização.
CREATE TABLE elt_2020.vra_convertida AS
SELECT UPPER(elt_2020.limpar_texto("Sigla ICAO Empresa Aérea")) AS companhia_icao,
       elt_2020.limpar_texto("Empresa Aérea") AS companhia_nome,
       elt_2020.limpar_texto("Número Voo") AS numero_voo,
       UPPER(elt_2020.limpar_texto("Código DI")) AS codigo_di,
       UPPER(elt_2020.limpar_texto("Código Tipo Linha")) AS tipo_linha,
       UPPER(elt_2020.limpar_texto("Modelo Equipamento")) AS modelo_equipamento,
       elt_2020.inteiro_seguro(elt_2020.limpar_texto("Número de Assentos")) AS numero_assentos,
       UPPER(elt_2020.limpar_texto("Sigla ICAO Aeroporto Origem")) AS aeroporto_origem,
       elt_2020.limpar_texto("Descrição Aeroporto Origem") AS descricao_origem,
       elt_2020.datahora_segura(elt_2020.limpar_texto("Partida Prevista")) AS partida_prevista,
       elt_2020.datahora_segura(elt_2020.limpar_texto("Partida Real")) AS partida_real,
       UPPER(elt_2020.limpar_texto("Sigla ICAO Aeroporto Destino")) AS aeroporto_destino,
       elt_2020.limpar_texto("Descrição Aeroporto Destino") AS descricao_destino,
       elt_2020.datahora_segura(elt_2020.limpar_texto("Chegada Prevista")) AS chegada_prevista,
       elt_2020.datahora_segura(elt_2020.limpar_texto("Chegada Real")) AS chegada_real,
       UPPER(elt_2020.limpar_texto("Situação Voo")) AS situacao_voo,
       elt_2020.limpar_texto("Justificativa") AS justificativa,
       elt_2020.data_segura(elt_2020.limpar_texto("Referência")) AS referencia,
       UPPER(elt_2020.limpar_texto("Situação Partida")) AS situacao_partida,
       UPPER(elt_2020.limpar_texto("Situação Chegada")) AS situacao_chegada,
       arquivo_origem, linha_origem, 2020::INTEGER AS ano_origem
FROM elt_2020.vra_classificada
WHERE ocorrencia_num = 1;

CREATE TABLE elt_2020.vra_avaliada AS
SELECT c.*, ARRAY_REMOVE(ARRAY[
    CASE WHEN companhia_icao IS NULL THEN 'COMPANHIA_AUSENTE' END,
    CASE WHEN aeroporto_origem IS NULL THEN 'AEROPORTO_ORIGEM_AUSENTE' END,
    CASE WHEN aeroporto_destino IS NULL THEN 'AEROPORTO_DESTINO_AUSENTE' END,
    CASE WHEN partida_prevista IS NULL THEN
        CASE WHEN elt_2020.limpar_texto(b."Partida Prevista") IS NULL
             THEN 'PARTIDA_PREVISTA_AUSENTE' ELSE 'PARTIDA_PREVISTA_INVALIDA' END END,
    CASE WHEN chegada_prevista IS NULL THEN
        CASE WHEN elt_2020.limpar_texto(b."Chegada Prevista") IS NULL
             THEN 'CHEGADA_PREVISTA_AUSENTE' ELSE 'CHEGADA_PREVISTA_INVALIDA' END END
], NULL) AS motivos_rejeicao
FROM elt_2020.vra_convertida c
JOIN elt_2020.vra_bruta b USING (arquivo_origem, linha_origem);

-- Preserva todos os campos originais e todos os motivos (uma linha por registro).
CREATE TABLE elt_2020.vra_rejeicoes AS
SELECT b.*, a.motivos_rejeicao
FROM elt_2020.vra_bruta b
JOIN elt_2020.vra_avaliada a USING (arquivo_origem, linha_origem)
WHERE cardinality(a.motivos_rejeicao) > 0;

CREATE TABLE elt_2020.vra_tratada AS
WITH medidas AS (
    SELECT a.*,
           EXTRACT(EPOCH FROM (partida_real - partida_prevista))/60 AS diferenca_partida_min,
           EXTRACT(EPOCH FROM (chegada_real - chegada_prevista))/60 AS diferenca_chegada_min
    FROM elt_2020.vra_avaliada a
    WHERE cardinality(motivos_rejeicao) = 0
)
SELECT *,
       CASE WHEN diferenca_partida_min IS NOT NULL THEN GREATEST(diferenca_partida_min,0) END AS atraso_partida_min,
       CASE WHEN diferenca_chegada_min IS NOT NULL THEN GREATEST(diferenca_chegada_min,0) END AS atraso_chegada_min,
       COALESCE(ABS(diferenca_partida_min)>1440,FALSE) AS partida_atipica_flag,
       COALESCE(ABS(diferenca_chegada_min)>1440,FALSE) AS chegada_atipica_flag
FROM medidas;

ALTER TABLE elt_2020.vra_tratada DROP COLUMN motivos_rejeicao;
ALTER TABLE elt_2020.vra_tratada ADD PRIMARY KEY (arquivo_origem, linha_origem);
ALTER TABLE elt_2020.vra_rejeicoes ADD PRIMARY KEY (arquivo_origem, linha_origem);

-- Campos opcionais inválidos não excluem o voo; ficam registrados para auditoria.
CREATE TABLE elt_2020.vra_anomalias AS
SELECT c.arquivo_origem, c.linha_origem, v.campo, v.valor_original,
       'VALOR_OPCIONAL_INVALIDO'::TEXT AS motivo
FROM elt_2020.vra_convertida c
JOIN elt_2020.vra_bruta b USING (arquivo_origem, linha_origem)
CROSS JOIN LATERAL (VALUES
    ('Partida Real', b."Partida Real", c.partida_real IS NULL),
    ('Chegada Real', b."Chegada Real", c.chegada_real IS NULL),
    ('Número de Assentos', b."Número de Assentos", c.numero_assentos IS NULL),
    ('Referência', b."Referência", c.referencia IS NULL)
) AS v(campo, valor_original, falhou)
WHERE elt_2020.limpar_texto(v.valor_original) IS NOT NULL AND v.falhou;

INSERT INTO elt_2020.vra_anomalias
SELECT arquivo_origem, linha_origem, 'Referência', referencia::TEXT, 'REFERENCIA_FORA_DE_2020'
FROM elt_2020.vra_convertida
WHERE referencia IS NOT NULL AND EXTRACT(YEAR FROM referencia) <> 2020;

CREATE TABLE elt_2020.resumo_qualidade AS
SELECT
    (SELECT COUNT(*) FROM elt_2020.vra_bruta) AS bruta,
    (SELECT COUNT(*) FROM elt_2020.vra_duplicatas_excluidas) AS duplicatas_excedentes,
    (SELECT COUNT(*) FROM elt_2020.vra_rejeicoes) AS rejeicoes,
    (SELECT COUNT(*) FROM elt_2020.vra_tratada) AS tratada,
    (SELECT COUNT(*) FROM elt_2020.vra_tratada WHERE partida_atipica_flag) AS partidas_atipicas,
    (SELECT COUNT(*) FROM elt_2020.vra_tratada WHERE chegada_atipica_flag) AS chegadas_atipicas,
    (SELECT COUNT(*) FROM elt_2020.vra_tratada WHERE partida_real IS NULL) AS partidas_reais_nulas,
    (SELECT COUNT(*) FROM elt_2020.vra_tratada WHERE chegada_real IS NULL) AS chegadas_reais_nulas,
    (SELECT COUNT(*) FROM elt_2020.vra_anomalias) AS ocorrencias_anomalias,
    CURRENT_TIMESTAMP AS executado_em;
"""

with conn:
    with conn.cursor() as cur:
        cur.execute(SQL_TRANSFORMACAO)
print("Camada tratada e tabelas de auditoria criadas por SQL.")

## 9. Validar o resultado
A primeira igualdade é a conferência principal:

**bruta = duplicatas excedentes + rejeições após deduplicação + tratada**

Também verificamos campos obrigatórios, unicidade da origem, preservação da carga e cálculo dos atrasos. Os testes de datas incluem ano bissexto, dia impossível e horário inválido.

As anomalias opcionais e os motivos de rejeição não são novas linhas de voos: um registro pode ter mais de um motivo.

In [ ]:
resumo = consultar("SELECT * FROM elt_2020.resumo_qualidade")
display(resumo.T.rename(columns={0: "Resultado"}))
r = resumo.iloc[0]
assert int(r["bruta"]) == total_csv, "A camada bruta diverge do ZIP."
assert r["bruta"] == r["duplicatas_excedentes"] + r["rejeicoes"] + r["tratada"], "Falha na reconciliação."

checagens = consultar("""
    SELECT
      NOT EXISTS (
        SELECT 1 FROM elt_2020.vra_tratada
        WHERE companhia_icao IS NULL OR aeroporto_origem IS NULL OR aeroporto_destino IS NULL
           OR partida_prevista IS NULL OR chegada_prevista IS NULL
      ) AS obrigatorios_ok,
      NOT EXISTS (
        SELECT 1 FROM elt_2020.vra_tratada t JOIN elt_2020.vra_rejeicoes r
        USING (arquivo_origem, linha_origem)
      ) AS conjuntos_separados,
      NOT EXISTS (
        SELECT 1 FROM elt_2020.vra_tratada t JOIN elt_2020.vra_duplicatas_excluidas d
        USING (arquivo_origem, linha_origem)
      ) AS sem_excedentes_na_tratada,
      NOT EXISTS (
        SELECT 1 FROM elt_2020.vra_tratada
        WHERE atraso_partida_min IS DISTINCT FROM
          CASE WHEN partida_real IS NOT NULL THEN GREATEST(EXTRACT(EPOCH FROM (partida_real-partida_prevista))/60,0) END
        OR atraso_chegada_min IS DISTINCT FROM
          CASE WHEN chegada_real IS NOT NULL THEN GREATEST(EXTRACT(EPOCH FROM (chegada_real-chegada_prevista))/60,0) END
      ) AS atrasos_ok,
      elt_2020.datahora_segura('29/02/2020 23:59') = TIMESTAMP '2020-02-29 23:59' AS bissexto_ok,
      elt_2020.datahora_segura('29/02/2019 23:59') IS NULL AS dia_invalido_rejeitado,
      elt_2020.datahora_segura('30/04/2020 24:01') IS NULL AS hora_invalida_rejeitada,
      elt_2020.datahora_segura('31/04/2020 12:00') IS NULL AS abril_invalido_rejeitado,
      elt_2020.inteiro_seguro('-1') IS NULL AS assento_negativo_rejeitado,
      elt_2020.inteiro_seguro('99999999999999999999') IS NULL AS inteiro_grande_tratado
""")
display(checagens.T.rename(columns={0: "Passou?"}))
assert checagens.iloc[0].all(), "Uma validação falhou; consulte a tabela acima."

# As contagens individuais também precisam continuar iguais após transformar.
carga_final = consultar("""
    SELECT arquivo_origem, COUNT(*) AS total FROM elt_2020.vra_bruta
    GROUP BY arquivo_origem ORDER BY arquivo_origem
""")
assert dict(zip(carga_final.arquivo_origem, carga_final.total)) == {
    item["arquivo_origem"]: item["registros"] for item in inventario
}
validacao_concluida = True
print("ELT 2020: carga, reconciliação e verificações de qualidade passaram nesta execução.")

## 10. Evidências para explicar no relatório
Leia os resultados: quantos registros foram descartados como repetição exata, quantos ficaram em rejeições e quantos foram tratados? Uma rejeição com dois campos faltantes aparece em dois motivos, mas conta como **um registro rejeitado**.

Estes resultados ainda precisam ser comparados com o ETL de 2020 do colega, usando os mesmos CSVs e as mesmas regras. Igualdade dos totais é necessária, mas a comparação ideal inclui os valores por `arquivo_origem + linha_origem`.

In [ ]:
CONSULTAS_EVIDENCIAS = {
    "mensal": """
      SELECT i.arquivo_origem, i.registros AS bruta,
        (SELECT COUNT(*) FROM elt_2020.vra_duplicatas_excluidas d WHERE d.arquivo_origem=i.arquivo_origem) AS duplicatas,
        (SELECT COUNT(*) FROM elt_2020.vra_rejeicoes r WHERE r.arquivo_origem=i.arquivo_origem) AS rejeicoes,
        (SELECT COUNT(*) FROM elt_2020.vra_tratada t WHERE t.arquivo_origem=i.arquivo_origem) AS tratada
      FROM elt_2020.inventario_arquivos i ORDER BY i.arquivo_origem
    """,
    "motivos_rejeicao": """
      SELECT motivo, COUNT(*) AS quantidade
      FROM elt_2020.vra_rejeicoes CROSS JOIN LATERAL unnest(motivos_rejeicao) AS m(motivo)
      GROUP BY motivo ORDER BY quantidade DESC, motivo
    """,
    "situacao_voo": """
      SELECT situacao_voo, COUNT(*) AS quantidade FROM elt_2020.vra_tratada
      GROUP BY situacao_voo ORDER BY quantidade DESC
    """,
    "atipicos_amostra": """
      SELECT arquivo_origem, linha_origem, numero_voo, partida_prevista, partida_real,
             chegada_prevista, chegada_real, diferenca_partida_min, diferenca_chegada_min
      FROM elt_2020.vra_tratada WHERE partida_atipica_flag OR chegada_atipica_flag
      ORDER BY GREATEST(ABS(diferenca_partida_min),ABS(diferenca_chegada_min)) DESC,
               arquivo_origem, linha_origem LIMIT 10
    """,
    "duplicatas_amostra": """
      SELECT arquivo_origem, linha_origem, arquivo_preservado, linha_preservada
      FROM elt_2020.vra_duplicatas_excluidas ORDER BY arquivo_origem, linha_origem LIMIT 10
    """
}
evidencias = {}
for titulo, consulta in CONSULTAS_EVIDENCIAS.items():
    print("\n" + titulo.replace("_", " ").upper())
    evidencias[titulo] = consultar(consulta)
    display(evidencias[titulo])
mensal = evidencias["mensal"]
assert (mensal.bruta == mensal.duplicatas + mensal.rejeicoes + mensal.tratada).all()

## 11. Gerar e baixar as evidências
Esta etapa cria `evidencias_elt_2020.zip` com resumo, inventário e hashes dos arquivos, contagens por mês, rejeições, duplicatas, anomalias e o SQL executado.

A tabela tratada completa fica no PostgreSQL. Para incluí-la no ZIP, altere `EXPORTAR_TRATADA_COMPLETA` para `True`. O arquivo será maior; não é necessário colocar a base inteira no GitHub.

Só gere a entrega final após a etapa 9 passar.

In [ ]:
from pathlib import Path
import json
import platform

assert globals().get("validacao_concluida", False), "Execute a validação da etapa 9 primeiro."
EXPORTAR_TRATADA_COMPLETA = False
pasta_saida = Path("/content/resultados_elt_2020")
pasta_saida.mkdir(parents=True, exist_ok=True)
arquivos_saida = []

def guardar_texto(nome, conteudo):
    destino = pasta_saida / nome
    destino.write_text(conteudo, encoding="utf-8")
    arquivos_saida.append(destino)

def exportar_consulta(nome, consulta):
    destino = pasta_saida / nome
    with conn:
        with conn.cursor() as cur:
            with destino.open("w", encoding="utf-8-sig", newline="") as arquivo:
                cur.copy_expert("COPY (" + consulta.strip().rstrip(";") +
                                ") TO STDOUT WITH (FORMAT CSV, HEADER TRUE)", arquivo)
    arquivos_saida.append(destino)

guardar_texto("transformacoes_elt_2020.sql", SQL_TRANSFORMACAO)
exportar_consulta("resumo_qualidade_2020.csv", "SELECT * FROM elt_2020.resumo_qualidade")
exportar_consulta("inventario_arquivos_2020.csv", "SELECT * FROM elt_2020.inventario_arquivos ORDER BY mes")
for nome, consulta in CONSULTAS_EVIDENCIAS.items():
    exportar_consulta(nome + "_2020.csv", consulta)
for tabela in ["vra_rejeicoes", "vra_duplicatas_excluidas", "vra_anomalias"]:
    exportar_consulta(tabela + "_2020.csv",
                     "SELECT * FROM elt_2020." + tabela + " ORDER BY arquivo_origem, linha_origem")
if EXPORTAR_TRATADA_COMPLETA:
    exportar_consulta("vra_tratada_2020.csv",
                     "SELECT * FROM elt_2020.vra_tratada ORDER BY arquivo_origem, linha_origem")

versao_postgres = consultar("SELECT version() AS versao").iloc[0]["versao"]
metadados = {
    "autor_adaptacao": "Gustavo José Arraes de Queiroz",
    "pipeline": "ELT", "ano_origem": ANO, "versao_pipeline": VERSAO_PIPELINE,
    "inicio_utc": inicio_execucao, "fim_utc": datetime.now(timezone.utc).isoformat(),
    "zip_origem": nome_zip, "sha256_zip": sha256_zip, "arquivos": inventario,
    "python": platform.python_version(), "postgresql": versao_postgres,
    "validacao": "Verificações da etapa 9 passaram; reconciliação mensal da etapa 10 passou.",
    "comparacao_etl": "Pendente de comparação com o resultado do colega."
}
guardar_texto("metadados_execucao_2020.json", json.dumps(metadados, ensure_ascii=False, indent=2))
guardar_texto("LEIA_ME.md", f"""# Evidências ELT 2020
Adaptação: Gustavo José Arraes de Queiroz. Base: notebook ELT 2019 do grupo.

Execução iniciada em {inicio_execucao} (UTC).
ZIP original: {nome_zip}
SHA-256: {sha256_zip}

Reconciliação: {int(r['bruta'])} brutos = {int(r['duplicatas_excedentes'])} duplicatas excedentes
+ {int(r['rejeicoes'])} rejeições + {int(r['tratada'])} tratados.

O SQL transforma somente após a carga dos valores originais no PostgreSQL.
Duplicatas são comparadas nas 20 colunas antes da padronização.
Sem horário real, o atraso permanece nulo.
Diferenças acima de 24 horas são sinalizadas e preservadas.
Rejeições conservam todos os valores originais e todos os motivos.
O inventário contém quantidade e SHA-256 de cada CSV, para comparar com o ETL.

Escopo: camada tratada e auditoria. Dimensões e fato compartilhadas não foram criadas.
A comparação completa entre ETL e ELT permanece pendente.
transformacoes_elt_2020.sql pressupõe a carga bruta criada pelo notebook.

Para o GitHub, selecione notebook, SQL, resumo e explicação dos resultados.
Não é necessário enviar os CSVs brutos ou a tabela tratada completa.
""")

caminho_zip = Path("/content/evidencias_elt_2020.zip")
with zipfile.ZipFile(caminho_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for arquivo in arquivos_saida:
        z.write(arquivo, arcname=arquivo.name)
print("Evidências geradas:", caminho_zip.name)
files.download(str(caminho_zip))

## 12. Evidências da execução individual
Após conferir os resultados, baixe o ZIP da célula anterior para guardar o SQL e as evidências da sessão.

**Para explicar sua parte:** Python leu e carregou os dados; o PostgreSQL fez a transformação. A tabela bruta permite auditar cada resultado. Duplicatas excedentes e rejeições têm destinos separados, e a reconciliação impede que uma linha desapareça sem explicação.

A carga dimensional e a comparação dos três anos são executadas separadamente pelo [guia da integração final](../../documentacao/INTEGRACAO_FINAL.md).